# 04 Clustering Resolution — Соусы

Цель: превратить pairwise predictions из `03_matching_comparison.ipynb` в графовые группы SKU.

- **Product family**: рёбра `exact_duplicate` + `same_product_different_pack`.
- **Pack group**: рёбра только `exact_duplicate` внутри family.
- Важно: текущий gold-set — это sampled pairs, а не полная кластерная разметка всех SKU. Поэтому метрики ниже — partial sanity-check по размеченным парам, а не окончательная cluster quality по всей категории.


## План

1. Прочитать `matching_predictions_sauces.csv` из notebook-3.
2. Выбрать лучший calibrated method по held-out `test` из `matching_summary_sauces.csv` или взять `DEDUP_CLUSTERING_METHOD`.
3. Построить true/predicted family-components и pack-components.
4. Посчитать pair-level linked/not-linked качество на held-out `test`.
5. Показать false links / missed links и сохранить component/pair-eval CSV для финального отчёта.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import sys

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    FAMILY_EDGE_LABELS,
    PACK_EDGE_LABELS,
    ComponentConfig,
    add_component_flags,
    build_components,
    component_size_summary,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


In [2]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
PREDICTIONS_PATH = DATA_DIR / "matching_predictions_sauces.csv"
SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
COMPONENTS_PATH = DATA_DIR / "clustering_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "clustering_pair_eval_sauces.csv"

METHOD_FROM_ENV = os.environ.get("DEDUP_CLUSTERING_METHOD")
EVAL_SPLIT = os.environ.get("DEDUP_CLUSTERING_EVAL_SPLIT", "test")

print(f"Predictions path: {PREDICTIONS_PATH}")
print(f"Summary path: {SUMMARY_PATH}")
print(f"Requested method: {METHOD_FROM_ENV or '<best calibrated test method>'}")
print(f"Eval split: {EVAL_SPLIT}")


Predictions path: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_predictions_sauces.csv
Summary path: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_summary_sauces.csv
Requested method: <best calibrated test method>
Eval split: test


In [3]:
def _load_predictions() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not PREDICTIONS_PATH.exists():
        raise FileNotFoundError(
            f"{PREDICTIONS_PATH} not found. Run notebooks/03_matching_comparison.ipynb first."
        )
    predictions = pd.read_csv(PREDICTIONS_PATH)
    summary = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else pd.DataFrame()
    return predictions, summary


def _select_method(predictions: pd.DataFrame, summary: pd.DataFrame) -> str:
    if METHOD_FROM_ENV:
        return METHOD_FROM_ENV
    if not summary.empty:
        candidates = summary[(summary["mode"].eq("calibrated")) & (summary["eval_split"].eq("test"))]
        if not candidates.empty:
            return str(candidates.sort_values("macro_f1", ascending=False).iloc[0]["method"])
    return str(predictions["method"].iloc[0])


predictions, matching_summary = _load_predictions()
selected_method = _select_method(predictions, matching_summary)
method_pairs = predictions[(predictions["method"].eq(selected_method)) & (predictions["mode"].eq("calibrated"))].copy()

if method_pairs.empty:
    raise ValueError(f"No calibrated predictions found for method={selected_method!r}")

print(f"Selected method: {selected_method}")
if not matching_summary.empty:
    display(matching_summary)
display(method_pairs["predicted_label"].value_counts().rename_axis("predicted_label").reset_index(name="pairs"))
display(pd.crosstab(method_pairs["eval_split"], method_pairs["predicted_label"]))


Selected method: rule_based_fuzzy


,method,mode,eval_split,threshold_high,pairs,macro_precision,macro_recall,macro_f1,exact_duplicate_precision,false_merge_count,false_merge_rate
0,rule_based_fuzzy,default_full_gold_set_sanity,all,0.82,379,0.591363,0.648863,0.606437,0.425532,89,0.234828
1,bi_encoder_zero_shot,default_full_gold_set_sanity,all,0.86,379,0.501348,0.582786,0.391180,0.262097,215,0.567282
2,rule_based_fuzzy,calibrated,dev,0.86,228,0.627863,0.626445,0.618742,0.517241,29,0.127193
3,rule_based_fuzzy,calibrated,test,0.86,151,0.619812,0.636099,0.606322,0.578947,28,0.185430
4,bi_encoder_zero_shot,calibrated,dev,1.00,228,0.758277,0.545412,0.511972,1.000000,24,0.105263
5,bi_encoder_zero_shot,calibrated,test,1.00,151,0.730000,0.539272,0.470139,1.000000,23,0.152318


,predicted_label,pairs
0,different_product,223
1,same_product_different_pack,108
2,exact_duplicate,48


predicted_label,different_product,exact_duplicate,same_product_different_pack
eval_split,,,
dev,141,29,58
test,82,19,50


In [4]:
def _node_catalog(pairs: pd.DataFrame) -> pd.DataFrame:
    left_cols = {
        "raw_record_id_a": "node_id",
        "marketplace_a": "marketplace",
        "sku_a": "sku",
        "title_a": "title",
        "brand_a": "brand",
        "unit_amount_a": "unit_amount",
        "total_amount_a": "total_amount",
        "multipack_count_a": "multipack_count",
    }
    right_cols = {key.replace("_a", "_b"): value for key, value in left_cols.items()}
    left = pairs[[column for column in left_cols if column in pairs.columns]].rename(columns=left_cols)
    right = pairs[[column for column in right_cols if column in pairs.columns]].rename(columns=right_cols)
    return pd.concat([left, right], ignore_index=True).drop_duplicates("node_id").reset_index(drop=True)


true_family_config = ComponentConfig(label_col="label", component_col="true_family_id")
pred_family_config = ComponentConfig(label_col="predicted_label", component_col="pred_family_id")
true_pack_config = ComponentConfig(label_col="label", component_col="true_pack_id")
pred_pack_config = ComponentConfig(label_col="predicted_label", component_col="pred_pack_id")

true_family_components = build_components(method_pairs, edge_labels=FAMILY_EDGE_LABELS, config=true_family_config)
pred_family_components = build_components(method_pairs, edge_labels=FAMILY_EDGE_LABELS, config=pred_family_config)
true_pack_components = build_components(method_pairs, edge_labels=PACK_EDGE_LABELS, config=true_pack_config)
pred_pack_components = build_components(method_pairs, edge_labels=PACK_EDGE_LABELS, config=pred_pack_config)

component_summary = pd.DataFrame([
    {"graph": "true_family", "components": true_family_components["true_family_id"].nunique(), "multi_node_components": int((component_size_summary(true_family_components, component_col="true_family_id")["nodes"] > 1).sum())},
    {"graph": "pred_family", "components": pred_family_components["pred_family_id"].nunique(), "multi_node_components": int((component_size_summary(pred_family_components, component_col="pred_family_id")["nodes"] > 1).sum())},
    {"graph": "true_pack", "components": true_pack_components["true_pack_id"].nunique(), "multi_node_components": int((component_size_summary(true_pack_components, component_col="true_pack_id")["nodes"] > 1).sum())},
    {"graph": "pred_pack", "components": pred_pack_components["pred_pack_id"].nunique(), "multi_node_components": int((component_size_summary(pred_pack_components, component_col="pred_pack_id")["nodes"] > 1).sum())},
])

display(component_summary)
display(component_size_summary(pred_family_components, component_col="pred_family_id").head(15))
display(component_size_summary(pred_pack_components, component_col="pred_pack_id").head(15))


,graph,components,multi_node_components
0,true_family,568,137
1,pred_family,560,144
2,true_pack,644,71
3,pred_pack,668,48


,pred_family_id,nodes
0,44,4
1,76,4
2,42,3
3,83,3
4,91,3
5,156,3
6,159,3
7,285,3
8,286,3
9,320,3


,pred_pack_id,nodes
0,8,2
1,10,2
2,14,2
3,15,2
4,18,2
5,29,2
6,37,2
7,39,2
8,40,2
9,58,2


In [5]:
pair_eval = add_component_flags(
    method_pairs,
    true_family_components,
    config=true_family_config,
    same_component_col="true_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_family_components,
    config=pred_family_config,
    same_component_col="pred_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    true_pack_components,
    config=true_pack_config,
    same_component_col="true_same_pack",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_pack_components,
    config=pred_pack_config,
    same_component_col="pred_same_pack",
)


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = frame[true_col].astype(bool)
    pred_link = frame[pred_col].astype(bool)
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "eval_split": frame["eval_split"].iloc[0] if frame["eval_split"].nunique() == 1 else "all",
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_links": tp,
        "false_links": fp,
        "missed_links": fn,
        "true_negative_links": tn,
    }


eval_pairs = pair_eval[pair_eval["eval_split"].eq(EVAL_SPLIT)].copy()
if eval_pairs.empty:
    eval_pairs = pair_eval.copy()

link_report = pd.DataFrame([
    _binary_link_report(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
    _binary_link_report(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
])
display(link_report)


,scope,eval_split,pairs,precision,recall,f1,true_positive_links,false_links,missed_links,true_negative_links
0,family,test,151,0.594203,0.694915,0.640625,41,28,18,64
1,pack,test,151,0.578947,0.379310,0.458333,11,8,18,114


In [6]:
def _show_link_examples(frame: pd.DataFrame, *, true_col: str, pred_col: str, title: str) -> None:
    false_links = frame[(~frame[true_col].astype(bool)) & frame[pred_col].astype(bool)].copy()
    missed_links = frame[frame[true_col].astype(bool) & (~frame[pred_col].astype(bool))].copy()
    columns = [
        "label",
        "predicted_label",
        "score",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    print(f"{title}: false links={len(false_links)}, missed links={len(missed_links)}")
    if not false_links.empty:
        display(false_links[[column for column in columns if column in false_links.columns]].head(12))
    if not missed_links.empty:
        display(missed_links[[column for column in columns if column in missed_links.columns]].head(12))


_show_link_examples(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", title="Family graph")
_show_link_examples(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", title="Pack graph")


Family graph: false links=28, missed links=18


,label,predicted_label,score,title_a,title_b,brand_a,brand_b,unit_amount_a,unit_amount_b,multipack_count_a,multipack_count_b
72,different_product,same_product_different_pack,0.604545,Соус Пад-Тай для лапши 5 шт х 80гр,соус Том Ям основа для супа 80гр,sen soy,sen soy,0.080,0.080,5.0,1.0
76,different_product,same_product_different_pack,0.668519,Корейская заправка для салата овощного из спаржи 3шт*60г,Заправка для салатов корейская для острой морковки 2/60г,чим-чим,чим-чим,0.060,0.060,3.0,1.0
102,different_product,same_product_different_pack,0.627000,"Соус Томатный низкокалорийный без сахара Bombbar, 2шт х 240г","Бомбар, Низкокалорийный соус 240 г, Барбекю",bombbar,bombbar,0.240,0.240,2.0,1.0
103,different_product,same_product_different_pack,0.513514,СТЕБЕЛЬ БАМБУКА Соус к мясу ПЭТ 280 гр 2 шт,Соус чили острый 280 гр 1 штука,стебель бамбука,стебель бамбука,0.280,0.280,2.0,1.0
108,different_product,exact_duplicate,0.891358,Соевый соус SanBonsai Терияки Stir Fry WOK с/б 275 гр,Соевый соус SanBonsai Терияки Stir Fry 300г,sanbonsai,sanbonsai,0.275,0.300,1.0,1.0
110,different_product,exact_duplicate,0.948450,"Уксус Monini Aceto Balsamico di Modena IGP Винный бальзамический, 500мл","Уксус Monini Aceto Balsamico винный бальзамический 6%, 250 мл",monini,monini,0.500,0.250,1.0,1.0
135,different_product,same_product_different_pack,0.596078,"Соус Calve Баварский медово-горчичный, 230 г 4 шт","Соус сливочно-чесночный, 230 г Calve",calve,calve,0.230,0.230,4.0,1.0
158,different_product,exact_duplicate,0.876190,"Низкокалорийный соус Mr.Djemius ZERO ""Сладкий чили"" 330г","Низкокалорийный соус Mr.Djemius ZERO ""Сацебели"" 330гр",mr. djemius zero,mr. djemius zero,0.330,0.330,1.0,1.0
160,different_product,exact_duplicate,0.876712,Соус Ореховый и Кимчи 470 мл 2 шт,Соус Ореховый и Сладкий чили 470 мл 2 шт,tamaki,tamaki,0.470,0.470,2.0,2.0
163,different_product,exact_duplicate,0.865823,Низкокалорийный соус без сахара Сальса 330г,"Соус Сладкий чили низкокалорийный, без сахара 330г",mr.djemius zero,mr. djemius zero,0.330,0.330,1.0,1.0


,label,predicted_label,score,title_a,title_b,brand_a,brand_b,unit_amount_a,unit_amount_b,multipack_count_a,multipack_count_b
32,exact_duplicate,different_product,0.697959,"Соус для шавермы Пикантье, 300 г","Соус чесночный для шавермы, 305 г",пикантье,пикантье,0.300,0.305,1.0,1.0
77,exact_duplicate,different_product,0.855000,"Соус бальзамический Monini, Balsamic Glaze, глазурь, 250 гр","Соус бальзамический Glaze, 250мл",monini,monin,0.250,0.250,1.0,1.0
83,exact_duplicate,different_product,0.847692,Аджика по-абхазски острая 2 шт по 120 г,"Аджика по-абхазски, 2 шт х 120 г Goldjick",goldjick,goldjick,0.120,0.120,2.0,2.0
86,exact_duplicate,different_product,0.760000,"Соус устричный Pearl River Bridge, 510 г","Соус устричный PRB 510 г, Китай",prb,pearl river bridge,0.510,0.510,1.0,1.0
92,exact_duplicate,different_product,0.828283,"Sen Soy основа для супа Premium ""Том ям/Tom yum"", 80 гр, пакет","SenSoy основа для супа Том ям \""TOM YUM\"", 80г",sen soy,sen soy premium,0.080,0.080,1.0,1.0
95,same_product_different_pack,different_product,0.798936,Рыбный соус AROY-D 240 г,Соус Рыбный 200 мл 4 шт,aroy-d,aroy-d,0.240,0.200,1.0,4.0
101,exact_duplicate,different_product,0.855000,Соус Кунжутный/Ореховый Tamaki 1 л.,Соус Кунжутный 1л,tamaki,tamaki,1.000,1.000,1.0,1.0
129,same_product_different_pack,different_product,0.791667,Соевый соус для мяса и рыбы 500 мл DAESANG,Соевый соус натурального брожжения для мяса и рыбы 200 мл,daesang,daesang,0.500,0.200,1.0,1.0
140,exact_duplicate,different_product,0.852756,Уксус красный винный нефильтрованный натуральный 6% ANDREA MILANO 500 мл ст/б Италия,"Organic DETO ANDREA MILANO Уксус красный винный нефильтрованный, 500 мл",andrea milano,andrea milano,0.500,0.500,1.0,1.0
157,exact_duplicate,different_product,0.809524,Соус Шрирача 475г,Острый соус Шрирача 475 г,uni-eagle,uni-eagle,0.475,0.475,1.0,1.0


Pack graph: false links=8, missed links=18


,label,predicted_label,score,title_a,title_b,brand_a,brand_b,unit_amount_a,unit_amount_b,multipack_count_a,multipack_count_b
4,same_product_different_pack,exact_duplicate,1.000000,"Уксус яблочный натуральный нефильтрованный Hamlitsch естественно мутный из штирийских яблок, с маткой 500 мл, Штирия","Уксус яблочный нефильтрованный Hamlitsch натуральный естественно мутный с маткой, из штирийских яблок 250 мл, Штирия",hamlitsch,hamlitsch,0.500,0.25,1.0,1.0
108,different_product,exact_duplicate,0.891358,Соевый соус SanBonsai Терияки Stir Fry WOK с/б 275 гр,Соевый соус SanBonsai Терияки Stir Fry 300г,sanbonsai,sanbonsai,0.275,0.30,1.0,1.0
110,different_product,exact_duplicate,0.948450,"Уксус Monini Aceto Balsamico di Modena IGP Винный бальзамический, 500мл","Уксус Monini Aceto Balsamico винный бальзамический 6%, 250 мл",monini,monini,0.500,0.25,1.0,1.0
158,different_product,exact_duplicate,0.876190,"Низкокалорийный соус Mr.Djemius ZERO ""Сладкий чили"" 330г","Низкокалорийный соус Mr.Djemius ZERO ""Сацебели"" 330гр",mr. djemius zero,mr. djemius zero,0.330,0.33,1.0,1.0
160,different_product,exact_duplicate,0.876712,Соус Ореховый и Кимчи 470 мл 2 шт,Соус Ореховый и Сладкий чили 470 мл 2 шт,tamaki,tamaki,0.470,0.47,2.0,2.0
163,different_product,exact_duplicate,0.865823,Низкокалорийный соус без сахара Сальса 330г,"Соус Сладкий чили низкокалорийный, без сахара 330г",mr.djemius zero,mr. djemius zero,0.330,0.33,1.0,1.0
240,same_product_different_pack,exact_duplicate,0.965986,"Натуральный сок лайма прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 200мл*6шт","Натуральный сок лимона прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 1л*6шт",азбука продуктов,азбука продуктов,0.200,1.00,6.0,6.0
270,same_product_different_pack,exact_duplicate,1.000000,Соус Томный барбекю и копчёная слива 1000 г,Соус Томный барбекю и копчёная слива 110 г,жар-соус,жар-соус,1.000,0.11,1.0,1.0


,label,predicted_label,score,title_a,title_b,brand_a,brand_b,unit_amount_a,unit_amount_b,multipack_count_a,multipack_count_b
32,exact_duplicate,different_product,0.697959,"Соус для шавермы Пикантье, 300 г","Соус чесночный для шавермы, 305 г",пикантье,пикантье,0.300,0.305,1.0,1.0
77,exact_duplicate,different_product,0.855000,"Соус бальзамический Monini, Balsamic Glaze, глазурь, 250 гр","Соус бальзамический Glaze, 250мл",monini,monin,0.250,0.250,1.0,1.0
83,exact_duplicate,different_product,0.847692,Аджика по-абхазски острая 2 шт по 120 г,"Аджика по-абхазски, 2 шт х 120 г Goldjick",goldjick,goldjick,0.120,0.120,2.0,2.0
86,exact_duplicate,different_product,0.760000,"Соус устричный Pearl River Bridge, 510 г","Соус устричный PRB 510 г, Китай",prb,pearl river bridge,0.510,0.510,1.0,1.0
92,exact_duplicate,different_product,0.828283,"Sen Soy основа для супа Premium ""Том ям/Tom yum"", 80 гр, пакет","SenSoy основа для супа Том ям \""TOM YUM\"", 80г",sen soy,sen soy premium,0.080,0.080,1.0,1.0
101,exact_duplicate,different_product,0.855000,Соус Кунжутный/Ореховый Tamaki 1 л.,Соус Кунжутный 1л,tamaki,tamaki,1.000,1.000,1.0,1.0
116,exact_duplicate,same_product_different_pack,0.865823,Заправка корейская для салата Фунчозы 3/60г,"Корейская заправка, ""Чим-Чим"", для фунчозы, 60г 3 шт",чим-чим,чим-чим,0.060,0.060,1.0,3.0
140,exact_duplicate,different_product,0.852756,Уксус красный винный нефильтрованный натуральный 6% ANDREA MILANO 500 мл ст/б Италия,"Organic DETO ANDREA MILANO Уксус красный винный нефильтрованный, 500 мл",andrea milano,andrea milano,0.500,0.500,1.0,1.0
157,exact_duplicate,different_product,0.809524,Соус Шрирача 475г,Острый соус Шрирача 475 г,uni-eagle,uni-eagle,0.475,0.475,1.0,1.0
241,exact_duplicate,same_product_different_pack,0.948780,Соус сладкий чили для курицы 470 мл - 2 шт,"Соус Сладкий чили для курицы Tamaki, 470 мл",tamaki,tamaki,0.470,0.470,2.0,1.0


In [7]:
node_catalog = _node_catalog(method_pairs)
components_export = (
    node_catalog
    .merge(pred_family_components, on="node_id", how="left")
    .merge(pred_pack_components, on="node_id", how="left")
    .merge(true_family_components, on="node_id", how="left")
    .merge(true_pack_components, on="node_id", how="left")
)

large_pred_families = component_size_summary(pred_family_components, component_col="pred_family_id")
large_pred_families = large_pred_families[large_pred_families["nodes"] > 1].head(10)
family_examples = components_export[components_export["pred_family_id"].isin(large_pred_families["pred_family_id"])].copy()
family_examples = family_examples.sort_values(["pred_family_id", "title"])

display(family_examples[["pred_family_id", "pred_pack_id", "marketplace", "sku", "brand", "title", "unit_amount", "total_amount", "multipack_count"]].head(40))

components_export.to_csv(COMPONENTS_PATH, index=False)
pair_eval.to_csv(PAIR_EVAL_PATH, index=False)
print(f"Saved components: {COMPONENTS_PATH} ({len(components_export)} rows)")
print(f"Saved pair eval: {PAIR_EVAL_PATH} ({len(pair_eval)} rows)")


,pred_family_id,pred_pack_id,marketplace,sku,brand,title,unit_amount,total_amount,multipack_count
377,42,108,Ozon,514069297,азбука продуктов,"Натуральный сок лайма прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 200мл",NaN,NaN,1.0
15,42,108,Ozon,194523239,азбука продуктов,"Натуральный сок лимона прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 1л",1.00,1.00,1.0
303,42,44,Ozon,1560707793,азбука продуктов,"Натуральный сок лимона прямого отжима АЗБУКА ПРОДУКТОВ основа для коктейлей и напитков, заправка для салата и соуса, приправа для рыбы и мяса 200мл*3шт",0.20,0.60,3.0
267,44,47,Ozon,1574067581,буздякский,Соус томатный Башкирский Буздякский - 3 шт x 670г,0.67,2.01,3.0
682,44,49,Ozon,1574082524,буздякский,Соус томатный Краснодарский Буздякский - 4 шт x 670г,NaN,NaN,4.0
626,44,48,Ozon,1574082401,буздякский,Соус томатный Краснодарский Буздякский - 6 шт x 670г,NaN,NaN,6.0
315,44,46,Ozon,1574057210,буздякский,Соус томатный Татарский Буздякский - 2 шт x 670г,0.67,1.34,2.0
115,76,306,WB,145093365,чим-чим,Заправка корейская для салата Фунчозы 3/60г,0.06,0.06,1.0
346,76,84,Ozon,1785181504,чим-чим,"Корейская заправка для моркови ""ЧИМ-ЧИМ"" 3 шт по 60 гр",0.06,0.18,3.0
341,76,83,Ozon,1785170876,чим-чим,"Корейская заправка для спаржи ""ЧИМ-ЧИМ"" 3 шт по 60 гр",0.06,0.18,3.0


Saved components: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/clustering_components_sauces.csv (716 rows)
Saved pair eval: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/clustering_pair_eval_sauces.csv (379 rows)


## Следующие шаги

- В `05_evaluation_report.ipynb` собрать итоговые таблицы: label distribution, matching summary, false-merge examples, clustering partial metrics.
- Для настоящей production-кластеризации нужно прогонять выбранный matcher по полному `candidates_sauces.csv`, а не только по 400 sampled gold-set pairs.
- Текущий граф показывает слабые места fusion: похожие вкусы/типы внутри одного бренда часто дают ложные family edges.
